In [1]:
!pip install -q datasets sentence-transformers openai

In [2]:
!pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 22.3 MB/s eta 0:00:00


In [3]:
import openai
from google.colab import userdata

llm_judge_client = openai.OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

In [4]:
# Removed: !pip install torch (torch is a dependency of sentence-transformers and is already installed.)

In [5]:
import torch
import json
import faiss
import numpy as np
import os
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def llm_judge_contamination(problem, ot_sample):
    global llm_judge_client
    prompt = f"""You are an expert at identifying contamination in datasets.
Given a benchmark problem and a potential contamination source, determine if the source is indeed a contamination for the problem.
A contamination means the source text is identical or nearly identical to the problem text, but the numerical values are exactly the same.

Respond with a single word: "YES" if it's contaminated, or "NO" if it's not.

Benchmark Problem: {problem}

Potential Contamination Source: {ot_sample}
"""
    response = llm_judge_client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = [
            {"role":"system","content":"You are an expert at identifying contamination in datasets. Only respond with a single word: \"YES\" or \"NO\". Consider a problem contaminated only if the source text is identical or nearly identical to the benchmark problem, and the numbers are exactly identical. It must have the exact same methodology and problem structure."},
            {"role":"user","content":prompt}
        ],
        max_tokens = 5,
        temperature = 0
    )
    return response.choices[0].message.content.strip().upper()


In [ ]:
# 1. Load Datasets and Preprocess Questions
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

print("Downloading datasets...")
ot_ds = load_dataset("open-thoughts/OpenThoughts-114k", split="train")
math_ds = load_dataset("HuggingFaceH4/MATH-500", split="test")

print("Preprocessing questions...")
ot_questions = [conv[0]['value'] for conv in ot_ds['conversations']]
math_questions = list(math_ds['problem']) # Convert to list

# Define paths for saved files
project_path = "./"
ot_embeddings_path = os.path.join(project_path, "ot_embeddings.npy")
ot_index_path = os.path.join(project_path, "ot_index.faiss")
ot_questions_path = os.path.join(project_path, "ot_questions.json")

print("Data loading and preprocessing complete.")

Running on: cuda


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00006.parquet:   0%|          | 0.00/250M [00:00<?, ?B/s]

data/train-00001-of-00006.parquet:   0%|          | 0.00/175M [00:00<?, ?B/s]

data/train-00002-of-00006.parquet:   0%|          | 0.00/173M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/113957 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Preprocessing questions...
Data loading and preprocessing complete.


In [ ]:
# 2. Load or Generate OpenThoughts Embeddings and FAISS Index
# Check if embeddings and index already exist

ot_embeddings = np.load(ot_embeddings_path)
gpu_index = faiss.read_index(ot_index_path)
if device == "cuda" and not isinstance(gpu_index, faiss.GpuIndex):
    res = faiss.StandardGpuResources()
    gpu_index = faiss.index_cpu_to_gpu(res, 0, gpu_index)
with open(ot_questions_path, "r") as f:
    ot_questions = json.load(f)
print("Existing embeddings and index loaded.")

# if os.path.exists(ot_embeddings_path) and os.path.exists(ot_index_path) and os.path.exists(ot_questions_path):
#     print("Loading existing OpenThoughts embeddings and FAISS index...")
#     ot_embeddings = np.load(ot_embeddings_path)
#     gpu_index = faiss.read_index(ot_index_path)
#     # If the index was saved as CPU index, convert it back to GPU if available
#     if device == "cuda" and not isinstance(gpu_index, faiss.GpuIndex):
#         res = faiss.StandardGpuResources()
#         gpu_index = faiss.index_cpu_to_gpu(res, 0, gpu_index)
#     with open(ot_questions_path, "r") as f:
#         ot_questions = json.load(f)
#     print("Existing embeddings and index loaded.")
# else:
#     print("Generating OpenThoughts embeddings and FAISS index...")
#     # 3. Model Initialization (SentenceTransformer for semantic embeddings)
#     model = SentenceTransformer('all-mpnet-base-v2', device=device)

#     # 4. Generate OpenThoughts Embeddings for FAISS
#     print(f"Embedding {len(ot_questions)} OpenThoughts items for FAISS...")
#     ot_embeddings = model.encode(
#         ot_questions,
#         batch_size=64,
#         show_progress_bar=True,
#         convert_to_numpy=True,
#         normalize_embeddings=True
#     )

#     # 5. Setup GPU-accelerated FAISS Index
#     dimension = ot_embeddings.shape[1]
#     res = faiss.StandardGpuResources()
#     index_flat = faiss.IndexFlatIP(dimension)
#     gpu_index = faiss.index_cpu_to_gpu(res, 0, index_flat)
#     gpu_index.add(ot_embeddings)

#     # Save the generated files for future use
#     print("Saving generated embeddings and index...")
#     faiss.write_index(faiss.index_gpu_to_cpu(gpu_index) if device == "cuda" else gpu_index, ot_index_path)
#     np.save(ot_embeddings_path, ot_embeddings)
#     with open(ot_questions_path, "w") as f:
#         json.dump(ot_questions, f)
#     print("Embeddings and index saved.")

Existing embeddings and index loaded.


In [ ]:
# 3. Generate MATH Embeddings for FAISS Search
print(f"Embedding {len(math_questions)} MATH-500 items for FAISS search...")
model = SentenceTransformer('all-mpnet-base-v2', device=device) # Re-initialize model to ensure it's available
math_embeddings = model.encode(
    math_questions,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Embedding 500 MATH-500 items for FAISS search...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
# 4. Generate TF-IDF Vectors
print("Generating TF-IDF vectors...")
tfidf_vectorizer = TfidfVectorizer(
    min_df=1,
    max_df=1.0,
    ngram_range=(1,3),
    token_pattern=r"(?u)\b[\w^+\-*/=]+\b"
)# Fit on combined corpus to ensure consistent vocabulary
tfidf_vectorizer.fit(ot_questions + math_questions)
ot_tfidf_vectors = tfidf_vectorizer.transform(ot_questions)
math_tfidf_vectors = tfidf_vectorizer.transform(math_questions)
print("TF-IDF vector generation complete.")

Generating TF-IDF vectors...
TF-IDF vector generation complete.


In [ ]:
# 5. Search for Semantic and TF-IDF Matches & LLM Judging
print("Searching for semantic and TF-IDF matches, and performing LLM judging...")
faiss_search_k = 5 # Number of top FAISS matches to consider for detailed analysis
faiss_threshold = 0.7 # Updated FAISS threshold as requested
tfidf_threshold = 0.7 # TF-IDF similarity threshold

report = []
for i in tqdm(range(len(math_questions)), desc="Processing Math500 problems"):
    current_math_problem = math_questions[i]

    # FAISS (Semantic) Search
    distances, indices = gpu_index.search(math_embeddings[i:i+1], k=faiss_search_k)
    faiss_matches = []
    for j in range(faiss_search_k):
        similarity = float(distances[0][j])
        if similarity >= faiss_threshold:
            faiss_matches.append({
                "ot_index": int(indices[0][j]),
                "similarity": round(similarity, 4),
                "ot_sample": ot_questions[indices[0][j]]
            })

    # TF-IDF Search
    # Calculate cosine similarity for the current math problem against all ot_questions
    tfidf_similarities = cosine_similarity(math_tfidf_vectors[i:i+1], ot_tfidf_vectors)[0]
    tfidf_matches = []
    # Get top K TF-IDF matches for reporting, similar to FAISS
    for j in np.argsort(tfidf_similarities)[::-1]: # Sort by similarity descending
        similarity = tfidf_similarities[j]
        if similarity >= tfidf_threshold:
            tfidf_matches.append({
                "ot_index": int(j),
                "similarity": round(float(similarity), 4),
                "ot_sample": ot_questions[j]
            })
        if len(tfidf_matches) >= faiss_search_k: # Limit TF-IDF matches to top K for consistency
            break

    # Combine potential matches for LLM judging (avoid duplicates and prioritize higher similarity if duplicated)
    all_potential_matches = {}
    for match in faiss_matches:
        all_potential_matches[match['ot_index']] = match # Key by ot_index
    for match in tfidf_matches:
        # Add if not already present or if similarity is higher (optional, but ensures best match is considered)
        if match['ot_index'] not in all_potential_matches or match['similarity'] > all_potential_matches[match['ot_index']]['similarity']:
            all_potential_matches[match['ot_index']] = match

    llm_judgments = []
    is_llm_contaminated_overall = "NO"
    if all_potential_matches:
        for ot_idx, match_data in all_potential_matches.items():
            llm_verdict = llm_judge_contamination(current_math_problem, match_data['ot_sample'])
            llm_judgments.append({
                "ot_index": ot_idx,
                "llm_verdict": llm_verdict,
                "source_text": match_data['ot_sample'] # Include source text for context
            })
            if llm_verdict == "YES":
                is_llm_contaminated_overall = "YES"

    report.append({
        "math500_idx": i,
        "benchmark_problem": current_math_problem,
        "semantic_matches": faiss_matches,
        "tfidf_matches": tfidf_matches,
        "llm_judgments": llm_judgments,
        "is_llm_contaminated_overall": is_llm_contaminated_overall,
        "is_contaminated_semantic": len(faiss_matches) > 0,
        "is_contaminated_tfidf": len(tfidf_matches) > 0
    })

# Save the report
report_filename = os.path.join(project_path, "contamination_report.json")
with open(report_filename, "w") as f:
    json.dump(report, f, indent=4)
print(f"Analysis Complete! Report saved to {report_filename}")

Searching for semantic and TF-IDF matches, and performing LLM judging...


Processing Math500 problems:   0%|          | 0/500 [00:00<?, ?it/s]

Analysis Complete! Report saved to ./contamination_report.json


In [ ]:
# Patch the existing report to fix the \BOXED{YES} formatting issue
for item in report:
    for judgment in item.get('llm_judgments', []):
        if 'YES' in judgment.get('llm_verdict', '').upper():
            item['is_llm_contaminated_overall'] = 'YES'

# Save the patched report back to disk
report_filename = os.path.join(project_path, "contamination_report.json")
with open(report_filename, "w") as f:
    json.dump(report, f, indent=4)

print("Report patched successfully! The is_llm_contaminated_overall flags have been fixed.")
print("You can now run the cells below to see the updated statistics and variance tests.")

Report patched successfully! The is_llm_contaminated_overall flags have been fixed.
You can now run the cells below to see the updated statistics and variance tests.


In [ ]:
# This cell contained stray code duplicated from cell 550453ab.
# It has been commented out to prevent execution errors.

# llm_judgments = []
# is_llm_contaminated_overall = "NO"
# if all_potential_matches:
#     for ot_idx, match_data in all_potential_matches.items():
#         llm_verdict = llm_judge_contamination(current_math_problem, match_data['ot_sample'])
#         llm_judgments.append({
#             "ot_index": ot_idx,
#             "llm_verdict": llm_verdict,
#             "source_text": match_data['ot_sample']
#         })
#         if llm_verdict == "YES":
#             is_llm_contaminated_overall = "YES"

In [ ]:
print("--- Contamination Report Statistics ---")
total_problems = len(report)
print(f"Total problems analyzed: {total_problems}")

# Count contamination by semantic search (FAISS)
semantic_contaminated_count = sum(1 for item in report if item['is_contaminated_semantic'])
print(f"Problems contaminated by Semantic Search (FAISS): {semantic_contaminated_count}")

# Count contamination by TF-IDF search
tfidf_contaminated_count = sum(1 for item in report if item['is_contaminated_tfidf'])
print(f"Problems contaminated by TF-IDF Search: {tfidf_contaminated_count}")

# Count contamination by LLM judging
llm_contaminated_count = sum(1 for item in report if item['is_llm_contaminated_overall'] == 'YES')
print(f"Problems contaminated by LLM Judging: {llm_contaminated_count}")

print("---------------------------------------")

--- Contamination Report Statistics ---
Total problems analyzed: 500
Problems contaminated by Semantic Search (FAISS): 195
Problems contaminated by TF-IDF Search: 15
Problems contaminated by LLM Judging: 8
---------------------------------------


In [ ]:
llm_contaminated_problems = [item for item in report if item['is_llm_contaminated_overall'] == 'YES']

llm_contaminated_filename = os.path.join(project_path, "llm_contaminated_report.json")
with open(llm_contaminated_filename, "w") as f:
    json.dump(llm_contaminated_problems, f, indent=4)

print(f"LLM contaminated problems saved to {llm_contaminated_filename}")


LLM contaminated problems saved to ./llm_contaminated_report.json


In [ ]:
display(llm_contaminated_problems)

[{'math500_idx': 189,
  'benchmark_problem': 'Five points $A$, $B$, $C$, $D$, and $O$ lie on a flat field.  $A$ is directly north of $O$, $B$ is directly west of $O$, $C$ is directly south of $O$, and $D$ is directly east of $O$. The  distance between $C$ and $D$ is 140 m.  A hot-air balloon is positioned in the air at $H$ directly above $O$. The balloon is held in place by four ropes $HA$, $HB$, $HC$, and $HD$.  Rope $HC$ has length 150 m and rope $HD$ has length 130 m. [asy]\nsize(250);\npair A, B, C, D, O, H, W, X, Y, Z;\nO=(0,0);\nA=(1,1);\nD=(1.5,-.3);\nB=(-1.5,.3);\nC=(-1,-1);\nH=(0,2.5);\nW=(5/3)*(A+D);\nX=(5/3)*(A+B);\nY=(-1)*(W);\nZ=(-1)*(X);\ndraw(W--X--Y--Z--W);\ndraw(A--C);\ndraw(B--D);\ndraw(O--H, linewidth(1));\ndraw(A--H, dashed);\ndraw(B--H, dashed);\ndraw(C--H, dashed);\ndraw(D--H, dashed);\ndot(A);\ndot(B);\ndot(C);\ndot(D);\ndot(O);\ndot(H);\nlabel("A", A, NE);\nlabel("B", B, SW);\nlabel("C", C, SE);\nlabel("D", D, NE);\nlabel("O", O, SE);\nlabel("H", H, NW);\n[/asy]

### Temperature-Sampling Variance Test
We will sample a few contaminated and uncontaminated problems. For each, we will generate 10 solutions at `temperature=1.0`. To measure "reasoning path variance", we will calculate the average pairwise cosine similarity of the 10 generated reasoning paths. High similarity means high consistency (low variance), which on complex problems may indicate recitation.

In [6]:
import random
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import json
import os

project_path = "./"
# Use the user-specified file for the report
report_filename = "/content/sample_30.jsonl"

# Assuming the file exists as confirmed by the user and is in JSON Lines format
report = []
with open(report_filename, "r") as f:
    for line in f:
        report.append(json.loads(line))

# 1. Select Samples
random.seed(42)
contaminated_samples = [p for p in report if p.get('contamination_type') != 'clean']
uncontaminated_samples = [p for p in report if p.get('contamination_type') == 'clean']

# Take up to 3 of each to keep API costs and time reasonable
num_samples = 3
test_contaminated = contaminated_samples
test_uncontaminated = uncontaminated_samples

test_cases = test_contaminated + test_uncontaminated
print(f"Selected {len(test_contaminated)} contaminated and {len(test_uncontaminated)} uncontaminated samples for variance testing.")

Selected 20 contaminated and 10 uncontaminated samples for variance testing.


In [7]:
!pip install rouge_score

In [ ]:
!pip install transformers

In [ ]:
!pip install --upgrade torch torchvision torchaudio
!pip install --upgrade sympy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.1 MB/s eta 0:00:00
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.10.0+cpu
    Uninstalling torchaudio-2.10.0+cpu:
      Successfully uninstalled torchaudio-2.10.0+cpu


In [14]:
dataset_to_model_map = {
    "tulu": "open-thoughts/OpenThinker-7B",
    "s1": "open-thoughts/OpenThinker-7B",
    "s2": "open-thoughts/OpenThinker-7B",
    "s3": "open-thoughts/OpenThinker-7B",
    "s4": "open-thoughts/OpenThinker-7B",
    "s5": "open-thoughts/OpenThinker-7B",
    "s6": "open-thoughts/OpenThinker-7B",
    "s7": "open-thoughts/OpenThinker-7B",
    "s8": "open-thoughts/OpenThinker-7B",
    "s9": "open-thoughts/OpenThinker-7B",
    "s10": "open-thoughts/OpenThinker-7B",
    "s11": "open-thoughts/OpenThinker-7B",
    "s12": "open-thoughts/OpenThinker-7B",
    "s13": "open-thoughts/OpenThinker-7B",
    "s14": "open-thoughts/OpenThinker-7B",
    "s15": "open-thoughts/OpenThinker-7B",
    "s16": "open-thoughts/OpenThinker-7B",
    "s17": "open-thoughts/OpenThinker-7B",
    "s18": "open-thoughts/OpenThinker-7B",
    "s19": "open-thoughts/OpenThinker-7B",
    "s20": "open-thoughts/OpenThinker-7B",
    "clean": "open-thoughts/OpenThinker-7B"
}
print("Defined dataset_to_model_map to use local OpenThinker model.")

Defined dataset_to_model_map to use local OpenThinker model.


In [ ]:
!pip uninstall -y torchcodec

Found existing installation: torchcodec 0.10.0
Uninstalling torchcodec-0.10.0:
  Successfully uninstalled torchcodec-0.10.0


In [12]:
!pip install -U bitsandbytes>=0.46.1

In [19]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import re
from collections import Counter
from rouge_score import rouge_scorer
import difflib

# Load the specific OpenThoughts model locally
hf_model_id = "open-thoughts/OpenThinker-7B"
print(f"Loading local model {hf_model_id} for variance testing in 4-bit...")

# Configure 4-bit quantization to save VRAM
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16
# )

tokenizer = AutoTokenizer.from_pretrained(hf_model_id)
hf_model = AutoModelForCausalLM.from_pretrained(
    hf_model_id,
    # quantization_config=bnb_config,
    device_map="auto"
)



Loading local model open-thoughts/OpenThinker-7B for variance testing in 4-bit...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [17]:
def generate_temperature_samples(problem, num_runs=5, model_id=None):
    # Note: model_id is kept in the signature so the downstream loops don't break,
    # but we are using the locally loaded hf_model for everything to avoid API limits.
    completions = []
    messages = [
        {"role": "system", "content": "You are a helpful math assistant. Solve the problem step-by-step."},
        {"role": "user", "content": problem}
    ]
    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_prompt, return_tensors="pt").to(hf_model.device)

    print(f"  -> Starting generation for {num_runs} paths locally...")
    for i in range(num_runs):
        try:
            print(f"     - Generating path {i+1}/{num_runs}...")
            with torch.no_grad():
                outputs = hf_model.generate(
                    **inputs,
                    max_new_tokens=1024,
                    temperature=1.0,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id
                )
            generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            completions.append(generated_text)
        except Exception as e:
            print(f"     ! Error generating path {i+1}: {e}")

    return completions

# METRIC 1: Semantic Consistency (Step-level Diversity = 1 - this)
def calculate_consistency_score(completions, encoder_model):
    if len(completions) < 2: return 0.0
    embeddings = encoder_model.encode(completions)
    sim_matrix = cosine_similarity(embeddings)
    n = len(completions)
    sum_sim = np.sum(np.triu(sim_matrix, k=1))
    return sum_sim / (n * (n - 1) / 2)

# METRIC 2: Answer Agreement Rate
def calculate_answer_agreement(completions):
    answers = []
    for text in completions:
        matches = re.findall(r'\\boxed{([^}]*)}', text)
        if matches:
            answers.append(matches[-1].strip())
        else:
            nums = re.findall(r'-?\d+\.?\d*', text)
            answers.append(nums[-1] if nums else "UNKNOWN")
    if not answers: return 0.0
    counts = Counter(answers)
    return counts.most_common(1)[0][1] / len(answers)

# METRIC 3: Lexical Overlap (ROUGE-L)
def calculate_lexical_overlap(completions):
    if len(completions) < 2: return 0.0
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = []
    n = len(completions)
    for i in range(n):
        for j in range(i+1, n):
            score = scorer.score(completions[i], completions[j])['rougeL'].fmeasure
            scores.append(score)
    return sum(scores) / len(scores)

# METRIC 4: Structural Fingerprinting (Exact Character Sequence Ratio)
def calculate_structural_fingerprint(completions):
    if len(completions) < 2: return 0.0
    ratios = []
    n = len(completions)
    for i in range(n):
        for j in range(i+1, n):
            sm = difflib.SequenceMatcher(None, completions[i], completions[j])
            ratios.append(sm.ratio())
    return sum(ratios) / len(ratios)


In [ ]:
import json
import os
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

runs_filename = "generated_runs.json"
num_runs_per_problem = 5

# Force regeneration by removing existing runs file if it exists and we're not explicitly skipping
if os.path.exists(runs_filename):
    print(f"Removing existing runs file: {runs_filename}")
    os.remove(runs_filename)

# 1. Generate and save runs (skip if already exists)
if not os.path.exists(runs_filename):
    print(f"Generating runs and saving to {runs_filename}...")
    all_generated_runs = []
    for case in tqdm(test_cases, desc="Generating Problems"):
        problem_text = case['problem'] # Use 'problem' field from the report
        status = "Contaminated" if case.get('contamination_type') != 'clean' else "Uncontaminated"
        source_dataset = case.get('source_dataset', 'none') # Get source_dataset

        # Determine the model ID based on the source_dataset
        # Default to OpenThinker if not found in map
        current_model_id = dataset_to_model_map.get(source_dataset, "open-thoughts/OpenThinker-7B")

        print(f"\n\n[Generating {status} Problem (Source: {source_dataset}, Model: {current_model_id})]: {problem_text[:50]}...")
        runs = generate_temperature_samples(problem_text, num_runs=num_runs_per_problem, model_id=current_model_id) # Pass model_id

        all_generated_runs.append({
            "benchmark_problem": problem_text,
            "status": status,
            "source_dataset": source_dataset, # Add source_dataset to results
            "model_used": current_model_id,   # Add model_used to results
            "runs": runs
        })
    with open(runs_filename, "w") as f:
        json.dump(all_generated_runs, f, indent=4)
    print("Generation complete and saved.")
else:
    print(f"Found existing runs in {runs_filename}. Skipping generation step.")

# 2. Load runs and evaluate metrics
print("Loading runs for evaluation...")
with open(runs_filename, "r") as f:
    loaded_runs = json.load(f)

# Ensure the sentence transformer model is loaded for semantic consistency scoring
if 'model' not in globals():
    print("Loading SentenceTransformer for evaluation...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer('all-mpnet-base-v2', device=device)

variance_results = []
for item in tqdm(loaded_runs, desc="Evaluating Problems"):
    problem_text = item['benchmark_problem']
    status = item['status']
    runs = item['runs']

    if len(runs) < 2:
        print(f"Not enough valid runs to calculate variance for: {problem_text[:30]}")
        continue

    semantic_consistency = calculate_consistency_score(runs, model)
    answer_agreement = calculate_answer_agreement(runs)
    lexical_overlap = calculate_lexical_overlap(runs)
    structural_fingerprint = calculate_structural_fingerprint(runs)

    variance_results.append({
        "Status": status,
        "Problem Snippet": problem_text[:80].replace('\n', ' ') + "...",
        "Answer Agreement Rate": round(answer_agreement, 3),
        "Semantic Consistency": round(semantic_consistency, 3),
        "Lexical Overlap (ROUGE)": round(lexical_overlap, 3),
        "Structural Overlap": round(structural_fingerprint, 3),
        "Num Paths": len(runs)
    })

print("\n--- FINAL VARIANCE ANALYSIS ---")
variance_df = pd.DataFrame(variance_results)
display(variance_df)

Removing existing runs file: generated_runs.json
Generating runs and saving to generated_runs.json...


Generating Problems:   0%|          | 0/6 [00:00<?, ?it/s]



[Generating Contaminated Problem (Source: tulu, Model: mistralai/Mistral-7B-Instruct-v0.2)]: A line is parameterized by
\[\begin{pmatrix} x \\ ...
  -> Starting generation for 5 paths via Hugging Face Inference API using model: mistralai/Mistral-7B-Instruct-v0.2...
     - Generating path 1/5...
     ! Error generating path 1: (Request ID: Root=1-69e9716a-5586beaa7592df7360d0d964;89b661a2-39d5-4eb9-bc9c-99e80cf83104)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.2' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}
     - Generating path 2/5...
     ! Error generating path 2: (Request ID: Root=1-69e9716a-03ae15c4084181910b859886;579f6d9d-9c3d-4513-8937-8e50efc8d3d4)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.2' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_no

Evaluating Problems:   0%|          | 0/6 [00:00<?, ?it/s]

Not enough valid runs to calculate variance for: A line is parameterized by
\[\
Not enough valid runs to calculate variance for: Let $\lambda$ be a constant, $
Not enough valid runs to calculate variance for: The following line is paramete
Not enough valid runs to calculate variance for: The volume of the cylinder sho
Not enough valid runs to calculate variance for: Find all values of $x$ that sa
Not enough valid runs to calculate variance for: The results of a cross-country

--- FINAL VARIANCE ANALYSIS ---


""


In [18]:
# Isolated test for the Hugging Face Inference API
print("Testing the updated generate_temperature_samples function...")

test_problem = "What is 2 + 2? Explain step-by-step."

# Testing a few models to see which ones are currently supported on the free tier
test_models = [
    "HuggingFaceH4/zephyr-7b-beta",
    "meta-llama/Llama-3.2-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct"
]

for model_id in test_models:
    print(f"\n--- Testing model: {model_id} ---")
    test_runs = generate_temperature_samples(test_problem, num_runs=1, model_id=model_id)
    if test_runs:
        print(f"Success! Generated response length: {len(test_runs[0])}")
        print("Snippet: ", test_runs[0][:100], "...")
    else:
        print("Failed to generate.")


Testing the updated generate_temperature_samples function...

--- Testing model: HuggingFaceH4/zephyr-7b-beta ---


NameError: name 'hf_model' is not defined